In [ ]:
# Load clean data
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_parquet("../data/processed/qualifying_clean.parquet")
print(df.shape)
df.head()

(25227, 30)


,Year,RoundNumber,EventName,Country,CircuitShortName,Driver,Team,LapTime_s,Compound,Sector1Time_s,...,Rainfall,TyreLife,FreshTyre,IsPersonalBest,BestLap_s,SessionFastest_s,DeltaToPersonalBest_s,DeltaToSessionFastest_s,IsStreetCircuit,Altitude_m
0,2021,1,Bahrain Grand Prix,Bahrain,Sakhir,VER,Red Bull Racing,90.499,SOFT,28.807,...,False,2.0,True,True,88.997,88.997,1.502,0.0,False,5
1,2021,1,Bahrain Grand Prix,Bahrain,Sakhir,VER,Red Bull Racing,110.360,SOFT,34.462,...,False,3.0,True,False,88.997,88.997,21.363,0.0,False,5
2,2021,1,Bahrain Grand Prix,Bahrain,Sakhir,VER,Red Bull Racing,90.318,MEDIUM,28.964,...,False,2.0,True,True,88.997,88.997,1.321,0.0,False,5
3,2021,1,Bahrain Grand Prix,Bahrain,Sakhir,VER,Red Bull Racing,115.675,MEDIUM,36.873,...,False,3.0,True,False,88.997,88.997,26.678,0.0,False,5
4,2021,1,Bahrain Grand Prix,Bahrain,Sakhir,VER,Red Bull Racing,91.820,SOFT,28.496,...,False,2.0,True,False,88.997,88.997,2.823,0.0,False,5


In [ ]:
# Season and circuit coverage
print("Seasons:", sorted(df["Year"].unique()))
print("Circuits:", df["EventName"].nunique())
print("Drivers:", df["Driver"].nunique())
print("Teams:", df["Team"].nunique())

Seasons: [2021, 2022, 2023, 2024, 2025, 2026]
Circuits: 29
Drivers: 36
Teams: 16


In [ ]:
# Lap time distribution by season
fig = px.box(
    df, x="Year", y="LapTime_s",
    title="Qualifying Lap Time Distribution by Season",
    labels={"LapTime_s": "Lap Time (seconds)", "Year": "Season"}
)
fig.show()

In [13]:
# Team qualifying pace distribution by season
fig = px.box(
    df,
    x="Team",
    y="LapTime_s",
    color="Team",
    facet_col="Year",
    facet_col_wrap=4,
    title="Qualifying Lap Time Distribution per Team per Season",
    labels={"LapTime_s": "Lap Time (s)", "Team": "Constructor"},
    height=900
)

fig.update_layout(
    showlegend=False,
    xaxis_tickangle=-45
)

# Hide x-axis team labels on all facets to reduce clutter (colour is the legend)
for axis in fig.layout:
    if axis.startswith("xaxis"):
        fig.layout[axis].showticklabels = False

fig.show()

In [14]:
# Delta to session fastest per team per circuit, one chart per season

# First compute each team's best lap per session (their fastest driver that round)
team_best = (
    df.groupby(["Year", "RoundNumber", "EventName", "Team"])["LapTime_s"]
    .min()
    .reset_index()
    .rename(columns={"LapTime_s": "TeamBestLap_s"})
)

# Fastest overall time per session
session_fastest = (
    df.groupby(["Year", "RoundNumber"])["LapTime_s"]
    .min()
    .reset_index()
    .rename(columns={"LapTime_s": "SessionFastest_s"})
)

# Merge and compute delta
team_delta = team_best.merge(session_fastest, on=["Year", "RoundNumber"])
team_delta["DeltaToFastest_s"] = team_delta["TeamBestLap_s"] - team_delta["SessionFastest_s"]

# Sort circuits by round number so x-axis is in race calendar order
team_delta = team_delta.sort_values(["Year", "RoundNumber"])

# Plot one chart per season
for year in sorted(team_delta["Year"].unique()):
    year_data = team_delta[team_delta["Year"] == year]

    fig = px.line(
        year_data,
        x="EventName",
        y="DeltaToFastest_s",
        color="Team",
        markers=True,
        title=f"{year} — Team Delta to Qualifying Fastest per Circuit",
        labels={
            "DeltaToFastest_s": "Delta to Fastest (s)",
            "EventName": "Circuit",
            "Team": "Constructor"
        },
        height=550
    )

    fig.update_layout(
        xaxis_tickangle=-45,
        xaxis_title="Circuit",
        yaxis_title="Delta to Fastest (s)",
        legend_title="Constructor",
        hovermode="x unified"
    )

    fig.show()

In [ ]:
# Delta to session fastest by circuit
# Shows which circuits produce closest/spread-out grids
delta_by_circuit = (
    df.groupby("EventName")["DeltaToSessionFastest_s"]
    .median()
    .sort_values(ascending=False)
    .reset_index()
)
fig = px.bar(
    delta_by_circuit, x="EventName", y="DeltaToSessionFastest_s",
    title="Median Delta to Session Fastest by Circuit",
    labels={"DeltaToSessionFastest_s": "Median Delta (s)", "EventName": "Circuit"}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# Track temperature vs lap time (scatter)
fig = px.scatter(
    df.sample(3000, random_state=42),
    x="TrackTemp", y="LapTime_s",
    color="Compound",
    title="Track Temperature vs Lap Time by Compound",
    labels={"TrackTemp": "Track Temp (°C)", "LapTime_s": "Lap Time (s)"},
    opacity=0.6
)
fig.show()

In [21]:
# Cell — Teams with the most session fastest qualifying times

team_poles = (
    df.groupby(["Year", "RoundNumber", "EventName"], group_keys=False)
    .apply(lambda x: x.loc[x["LapTime_s"].idxmin(), "Team"], include_groups=False)
    .reset_index()
    .rename(columns={0: "FastestTeam"})
)

team_pole_counts = (
    team_poles["FastestTeam"]
    .value_counts()
    .reset_index()
    .rename(columns={"FastestTeam": "Team", "count": "PoleCount"})
)

fig = px.bar(
    team_pole_counts,
    x="Team",
    y="PoleCount",
    color="Team",
    title="Teams with the Most Pole Positions (2018–2026)",
    labels={"Team": "Constructor", "PoleCount": "Number of Pole Positions"},
    text="PoleCount"
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

In [20]:
# Drivers with the most session fastest qualifying times

driver_poles = (
    df.groupby(["Year", "RoundNumber", "EventName"], group_keys=False)
    .apply(lambda x: x.loc[x["LapTime_s"].idxmin(), "Driver"], include_groups=False)
    .reset_index()
    .rename(columns={0: "FastestDriver"})
)

driver_pole_counts = (
    driver_poles["FastestDriver"]
    .value_counts()
    .reset_index()
    .rename(columns={"FastestDriver": "Driver", "count": "PoleCount"})
    .head(15)
)

fig = px.bar(
    driver_pole_counts,
    x="Driver",
    y="PoleCount",
    color="Driver",
    title="Top 15 Drivers with Most Pole Positions (2018–2026)",
    labels={"Driver": "Driver", "PoleCount": "Number of Pole Positions"},
    text="PoleCount"
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, xaxis_tickangle=-30)
fig.show()

In [ ]:
# Top 10 most represented drivers
top_drivers = df["Driver"].value_counts().head(10)
fig = px.bar(
    top_drivers, title="Top 10 Drivers by Lap Count",
    labels={"value": "Lap Count", "index": "Driver"}
)
fig.show()

In [ ]:
# Compound usage breakdown
fig = px.pie(
    df, names="Compound",
    title="Tyre Compound Distribution in Qualifying"
)
fig.show()

In [ ]:
# Check delta distribution (model target health check)
fig = px.histogram(
    df, x="DeltaToSessionFastest_s", nbins=80,
    title="Distribution of Delta to Session Fastest",
    labels={"DeltaToSessionFastest_s": "Delta to Fastest (s)"}
)
fig.show()

In [ ]:
# Missingness in the cleaned dataset
null_summary = df.isnull().sum().sort_values(ascending=False)
null_summary = null_summary[null_summary > 0]
print("Remaining nulls after cleaning:")
print(null_summary)

Remaining nulls after cleaning:
SpeedFL          7800
SpeedI1           151
SpeedST           141
Sector2Time_s      21
SpeedI2            18
Sector3Time_s      12
Sector1Time_s      11
dtype: int64


In [ ]:
# Street circuit vs permanent circuit pace spread
fig = px.box(
    df, x="IsStreetCircuit", y="DeltaToSessionFastest_s",
    title="Pace Spread: Street Circuits vs Permanent Circuits",
    labels={"IsStreetCircuit": "Street Circuit", "DeltaToSessionFastest_s": "Delta to Fastest (s)"}
)
fig.show()

In [26]:
# Driver performance: street circuits vs permanent circuits

# Get each driver's best lap per session
driver_session_best = (
    df.groupby(["Year", "RoundNumber", "EventName", "Driver", "IsStreetCircuit"])["LapTime_s"]
    .min()
    .reset_index()
    .rename(columns={"LapTime_s": "DriverBest_s"})
)

# Get session fastest for that round
session_fastest = (
    df.groupby(["Year", "RoundNumber"])["LapTime_s"]
    .min()
    .reset_index()
    .rename(columns={"LapTime_s": "SessionFastest_s"})
)

# Merge and compute delta
driver_circuit_delta = driver_session_best.merge(session_fastest, on=["Year", "RoundNumber"])
driver_circuit_delta["DeltaToFastest_s"] = driver_circuit_delta["DriverBest_s"] - driver_circuit_delta["SessionFastest_s"]

# Average delta per driver per circuit type
driver_avg = (
    driver_circuit_delta.groupby(["Driver", "IsStreetCircuit"])["DeltaToFastest_s"]
    .median()
    .reset_index()
)

driver_avg["CircuitType"] = driver_avg["IsStreetCircuit"].map({True: "Street Circuit", False: "Permanent Circuit"})

# Pivot so we can sort by the difference between the two
pivot = driver_avg.pivot(index="Driver", columns="CircuitType", values="DeltaToFastest_s").dropna()
pivot["StreetAdvantage"] = pivot["Permanent Circuit"] - pivot["Street Circuit"]
pivot = pivot.sort_values("StreetAdvantage", ascending=False).reset_index()

# Keep only drivers with enough appearances to be meaningful
min_sessions = 10
session_counts = driver_circuit_delta.groupby("Driver")["RoundNumber"].count()
qualified_drivers = session_counts[session_counts >= min_sessions].index
pivot = pivot[pivot["Driver"].isin(qualified_drivers)]

# Melt back for plotting
plot_data = pivot.melt(
    id_vars="Driver",
    value_vars=["Street Circuit", "Permanent Circuit"],
    var_name="CircuitType",
    value_name="MedianDeltaToFastest_s"
)

fig = px.bar(
    plot_data,
    x="Driver",
    y="MedianDeltaToFastest_s",
    color="CircuitType",
    barmode="group",
    title="Driver Median Delta to Fastest — Street Circuits vs Permanent Circuits (2018–2026)",
    labels={
        "MedianDeltaToFastest_s": "Median Delta to Session Fastest (s)",
        "Driver": "Driver",
        "CircuitType": "Circuit Type"
    },
    color_discrete_map={
        "Street Circuit": "#EF553B",
        "Permanent Circuit": "#636EFA"
    },
    height=600
)

fig.update_layout(
    xaxis_tickangle=-45,
    xaxis_title="Driver",
    yaxis_title="Median Delta to Fastest (s)",
    hovermode="x unified",
    legend_title="Circuit Type"
)

fig.show()

In [25]:
# Street circuit specialists: ranked by advantage over permanent circuits

fig = px.bar(
    pivot.head(30),  # Top 30 most street-advantaged drivers
    x="Driver",
    y="StreetAdvantage",
    color="StreetAdvantage",
    color_continuous_scale="RdYlGn",
    title="Street Circuit Specialists — Gap Between Permanent and Street Circuit Delta (2018–2026)",
    labels={
        "StreetAdvantage": "Street Advantage (s)\n(positive = better on streets)",
        "Driver": "Driver"
    },
    height=500
)

fig.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.4)
fig.update_layout(xaxis_tickangle=-45, coloraxis_showscale=False)
fig.show()